# GPT-1 기반 한국어 → 영어 번역기

## Transformer vs GPT-1 아키텍처 변경사항

### 1. 전체 구조 변경
| 구분 | Transformer | GPT-1 |
|------|-------------|-------|
| 구조 | Encoder-Decoder | **Decoder-only** |
| 인코더 | 있음 | **없음** |
| 디코더 레이어 | Self-Attention + Cross-Attention + FFN | **Self-Attention + FFN** |

### 2. 어텐션 메커니즘 변경
| 구분 | Transformer | GPT-1 |
|------|-------------|-------|
| Self-Attention | 인코더/디코더 각각 사용 | **디코더에서만 반복 사용** |
| Cross-Attention | 디코더에서 인코더 출력 참조 | **없음 (제거)** |
| Masking | Encoder: 양방향 / Decoder: 단방향 | **Look-ahead Mask만 사용 (단방향)** |

### 3. 위치 정보 인코딩 변경
| 구분 | Transformer | GPT-1 |
|------|-------------|-------|
| 방식 | Sinusoidal Positional Encoding (고정) | **Learnable Positional Embedding (학습)** |
| 구현 | sin/cos 함수로 계산 | **nn.Embedding으로 학습** |

### 4. 활성화 함수 변경
| 구분 | Transformer | GPT-1 |
|------|-------------|-------|
| FFN 활성화 함수 | ReLU | **GELU** |

### 5. 입력 데이터 형태 변경
| 구분 | Transformer | GPT-1 |
|------|-------------|-------|
| 입력 형태 | 소스/타겟 분리 | **소스 + 구분자 + 타겟 연결** |
| 형식 | src → encoder, tgt → decoder | **[KO] + [SEP] + [EN] 단일 시퀀스** |

## 라이브러리 버전 확인

In [ ]:
import torch
import sentencepiece
import platform

print('PyTorch      :', torch.__version__)
print('SentencePiece:', sentencepiece.__version__)
print('Python       :', platform.python_version())
print('CUDA  사용 가능:', torch.cuda.is_available())
print('MPS   사용 가능:', torch.backends.mps.is_available())

---
## Step 1. 데이터 다운로드
> **출처**: [jungyeul/korean-parallel-corpora](https://github.com/jungyeul/korean-parallel-corpora)

In [ ]:
import os

DATA_DIR = os.path.expanduser('~/work/transformer/data')
os.makedirs(DATA_DIR, exist_ok=True)

url = 'https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz'
gz_path  = os.path.join(DATA_DIR, 'korean-english-park.train.tar.gz')
tar_path = os.path.join(DATA_DIR, 'korean-english-park.train.tar')

if not os.path.exists(os.path.join(DATA_DIR, 'korean-english-park.train.ko')):
    os.system(f'wget -q -O {gz_path} {url}')
    os.system(f'gzip -d -f {gz_path}')
    os.system(f'tar -xvf {tar_path} -C {DATA_DIR}')
    print('다운로드 및 압축 해제 완료')
else:
    print('이미 데이터가 존재합니다.')

KO_FILE = os.path.join(DATA_DIR, 'korean-english-park.train.ko')
EN_FILE = os.path.join(DATA_DIR, 'korean-english-park.train.en')
print('KO:', KO_FILE)
print('EN:', EN_FILE)

---
## Step 2. 데이터 정제 및 토큰화
### 1. 중복 데이터 제거

In [ ]:
with open(KO_FILE, 'r', encoding='utf-8') as f:
    ko_lines = f.readlines()
with open(EN_FILE, 'r', encoding='utf-8') as f:
    en_lines = f.readlines()

seen = set()
cleaned_corpus = []

for ko, en in zip(ko_lines, en_lines):
    pair = (ko.strip(), en.strip())
    if pair not in seen:
        seen.add(pair)
        cleaned_corpus.append(pair)

print(f'원본 데이터 수    : {len(ko_lines):,}')
print(f'중복 제거 후 수   : {len(cleaned_corpus):,}')

### 2. 정제 함수 정의

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z가-힣\s.,!?]', '', text)
    text = re.sub(r'([.,!?])', r' \1 ', text)
    text = text.strip()
    return text

print(clean_text('오바마는 대통령이다.'))
print(clean_text('Obama is the President!'))

### 3. 토큰화 (SentencePiece)

**[GPT-1 변경사항]** GPT-1은 단일 시퀀스로 입력을 처리하므로, 한국어와 영어를 하나의 토크나이저로 통합합니다.

| 특수 토큰 | ID | 설명 |
|-----------|----|------|
| `<PAD>`   | 0  | 패딩 |
| `<BOS>`   | 1  | 시퀀스 시작 |
| `<EOS>`   | 2  | 시퀀스 끝 |
| `<UNK>`   | 3  | 미등록 토큰 |
| `<SEP>`   | 4  | **구분자 (한국어↔영어 구분)** |

In [ ]:
import sentencepiece as spm

kor_corpus = [clean_text(pair[0]) for pair in cleaned_corpus]
eng_corpus = [clean_text(pair[1]) for pair in cleaned_corpus]

# [GPT-1 변경사항] 한국어+영어 통합 코퍼스 생성 (단일 토크나이저용)
# Decoder-only 모델은 소스와 타겟을 하나의 시퀀스로 처리하므로 통합 토크나이저 사용
combined_corpus = kor_corpus + eng_corpus

with open('combined_corpus.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(combined_corpus))

# [GPT-1 변경사항] 통합 토크나이저 + SEP 토큰 추가
def generate_tokenizer(corpus_file, vocab_size=20000, model_prefix='tokenizer'):
    """SentencePiece 토크나이저 학습 후 반환"""
    spm.SentencePieceTrainer.Train(
        input=corpus_file,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        pad_id=0,   # <PAD>
        bos_id=1,   # <BOS>
        eos_id=2,   # <EOS>
        unk_id=3,   # <UNK>
        user_defined_symbols='<SEP>',  # [GPT-1] 소스-타겟 구분자
    )
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load(f'{model_prefix}.model')
    return tokenizer

# [GPT-1 변경사항] 단일 통합 토크나이저 사용
tokenizer = generate_tokenizer('combined_corpus.txt', model_prefix='gpt_tokenizer', vocab_size=25000)

# SEP 토큰 ID 확인
SEP_ID = tokenizer.PieceToId('<SEP>')
BOS_ID = tokenizer.bos_id()
EOS_ID = tokenizer.eos_id()
PAD_ID = tokenizer.pad_id()

print(f'Vocab size: {tokenizer.GetPieceSize():,}')
print(f'SEP_ID: {SEP_ID}, BOS_ID: {BOS_ID}, EOS_ID: {EOS_ID}, PAD_ID: {PAD_ID}')
print('KO sample:', tokenizer.EncodeAsIds('오바마는 대통령이다 .'))
print('EN sample:', tokenizer.EncodeAsIds('obama is the president .'))

### 4. GPT-1용 데이터 전처리

**[GPT-1 변경사항]** Decoder-only 모델의 입력 형태:
- Transformer: `Encoder(한국어)` + `Decoder(영어)` 분리 입력
- **GPT-1**: `[BOS] + 한국어 + [SEP] + 영어 + [EOS]` 단일 시퀀스

이렇게 하면 모델이 한국어 컨텍스트를 보고 영어를 생성하는 방식으로 학습됩니다.

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

MAX_LEN = 100  # [GPT-1] 소스+타겟 합친 길이이므로 더 길게 설정

# [GPT-1 변경사항] 소스와 타겟을 하나의 시퀀스로 연결
# 형식: [BOS] + 한국어 토큰 + [SEP] + 영어 토큰 + [EOS]
gpt_sequences = []
sep_positions = []  # SEP 위치 저장 (추론 시 필요)

for ko, en in zip(kor_corpus, eng_corpus):
    ko_ids = tokenizer.EncodeAsIds(ko)
    en_ids = tokenizer.EncodeAsIds(en)
    
    # [BOS] + 한국어 + [SEP] + 영어 + [EOS]
    sequence = [BOS_ID] + ko_ids + [SEP_ID] + en_ids + [EOS_ID]
    
    if len(sequence) <= MAX_LEN:
        gpt_sequences.append(torch.tensor(sequence, dtype=torch.long))
        sep_positions.append(len(ko_ids) + 1)  # BOS 다음부터 SEP까지의 위치

# 패딩 처리
gpt_train = pad_sequence(gpt_sequences, batch_first=True, padding_value=PAD_ID)

print(f'필터링 후 데이터 수: {len(gpt_sequences):,}')
print(f'gpt_train shape   : {gpt_train.shape}')
print(f'\n예시 시퀀스 (첫 번째):')
print(f'  토큰 ID: {gpt_train[0][:20].tolist()}...')
print(f'  디코딩: {tokenizer.DecodeIds(gpt_train[0][:20].tolist())}')

---
## Step 3. GPT-1 모델 설계

### [변경사항] Positional Embedding (기존: Positional Encoding)

**Transformer**: sin/cos 함수를 사용한 고정된 위치 인코딩

**GPT-1**: 학습 가능한 위치 임베딩 (nn.Embedding 사용)

In [ ]:
import torch.nn as nn

# ============================================================================
# [GPT-1 변경사항] Positional Encoding → Positional Embedding
# 
# 기존 Transformer:
#   - sin/cos 함수로 계산된 고정 위치 인코딩
#   - def positional_encoding(pos, d_model): ...
#
# GPT-1:
#   - 학습 가능한 위치 임베딩 (nn.Embedding)
#   - 모델 학습 과정에서 위치 정보도 함께 학습됨
# ============================================================================

class PositionalEmbedding(nn.Module):
    """[GPT-1] 학습 가능한 위치 임베딩
    
    Transformer의 sinusoidal positional encoding과 달리,
    GPT-1은 위치 정보를 학습 가능한 파라미터로 처리합니다.
    """
    def __init__(self, max_len, d_model):
        super().__init__()
        # [GPT-1] nn.Embedding으로 위치 임베딩 생성 (학습됨)
        self.pos_embed = nn.Embedding(max_len, d_model)
        
    def forward(self, x):
        """x: (batch_size, seq_len, d_model)"""
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        return self.pos_embed(positions)

print('PositionalEmbedding 클래스 정의 완료')

### [변경사항] PoswiseFeedForwardNet (ReLU → GELU)

**Transformer**: ReLU 활성화 함수 사용

**GPT-1**: GELU (Gaussian Error Linear Unit) 활성화 함수 사용

In [ ]:
# ============================================================================
# [GPT-1 변경사항] ReLU → GELU 활성화 함수
# 
# 기존 Transformer:
#   self.relu = nn.ReLU()
#   out = self.relu(self.fc1(x))
#
# GPT-1:
#   self.gelu = nn.GELU()
#   out = self.gelu(self.fc1(x))
#
# GELU는 입력값에 따라 부드럽게 활성화되어 더 나은 성능을 보임
# ============================================================================

class PoswiseFeedForwardNet(nn.Module):
    """Position-wise Feed Forward Network
    
    [GPT-1 변경] ReLU 대신 GELU 활성화 함수 사용
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1  = nn.Linear(d_model, d_ff)
        self.fc2  = nn.Linear(d_ff, d_model)
        # [GPT-1 변경] ReLU → GELU
        self.gelu = nn.GELU()  # 기존: self.relu = nn.ReLU()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        # [GPT-1 변경] GELU 활성화 함수 사용
        out = self.gelu(self.fc1(x))  # 기존: self.relu(self.fc1(x))
        out = self.fc2(out)
        return self.norm(out + residual)

print('PoswiseFeedForwardNet 클래스 정의 완료 (GELU 사용)')

### Multi-Head Self-Attention

**[GPT-1]** Self-Attention만 사용 (Cross-Attention 없음)

In [ ]:
# ============================================================================
# [GPT-1] Self-Attention만 사용
# 
# 기존 Transformer 디코더:
#   - Self-Attention (자기 참조)
#   - Cross-Attention (인코더 출력 참조) ← GPT-1에서는 제거됨
#
# GPT-1:
#   - Self-Attention만 반복 사용
#   - 인코더가 없으므로 Cross-Attention 불필요
# ============================================================================

class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention
    
    [GPT-1] Self-Attention만 사용 (Cross-Attention 없음)
    Q, K, V가 모두 동일한 입력에서 생성됨
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            # [GPT-1] Look-ahead mask 적용 (미래 토큰 참조 방지)
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        return torch.matmul(attn, V), attn

    def forward(self, x, mask=None):
        """[GPT-1] Self-Attention: Q=K=V=x
        
        기존 Transformer 디코더의 forward(Q, K, V, mask)와 달리
        GPT-1은 단일 입력 x만 받음
        """
        residual = x
        B = x.size(0)
        # [GPT-1] Q, K, V 모두 동일한 x에서 생성
        Q = self.W_Q(x).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        out, attn = self.scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        out = self.W_O(out)
        return self.norm(out + residual), attn

print('MultiHeadAttention 클래스 정의 완료 (Self-Attention only)')

### [변경사항] GPT-1 Decoder Block

**Transformer Decoder**: Self-Attention → Cross-Attention → FFN

**GPT-1 Block**: Self-Attention → FFN (Cross-Attention 제거)

In [ ]:
# ============================================================================
# [GPT-1 변경사항] Decoder Layer 구조 변경
# 
# 기존 Transformer Decoder Layer:
#   1. Masked Self-Attention
#   2. Cross-Attention (인코더 출력 참조) ← 제거됨
#   3. Feed Forward Network
#
# GPT-1 Block:
#   1. Masked Self-Attention (Look-ahead mask)
#   2. Feed Forward Network (GELU)
# ============================================================================

class GPTBlock(nn.Module):
    """GPT-1 Transformer Block
    
    [GPT-1 변경]
    - Cross-Attention 제거 (인코더 없음)
    - Self-Attention + FFN 구조
    """
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        # [GPT-1] Self-Attention만 사용 (Cross-Attention 제거)
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        # 기존 Transformer: self.cross_attn = MultiHeadAttention(...) ← 제거됨
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)  # GELU 사용
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """[GPT-1] 단일 입력, Look-ahead mask 적용
        
        기존 Transformer Decoder:
            forward(x, enc_out, src_mask, tgt_mask)
        GPT-1:
            forward(x, mask)  # enc_out, src_mask 불필요
        """
        # Self-Attention with Look-ahead mask
        out, self_attn = self.self_attn(x, mask)
        # 기존: out, cross_attn = self.cross_attn(out, enc_out, src_mask) ← 제거
        out = self.dropout(out)
        out = self.ffn(out)
        return out, self_attn

print('GPTBlock 클래스 정의 완료 (Cross-Attention 제거)')

### [변경사항] Masking - Look-ahead Mask만 사용

In [ ]:
# ============================================================================
# [GPT-1 변경사항] Masking 전략
# 
# 기존 Transformer:
#   - Encoder: Padding mask (양방향 참조 가능)
#   - Decoder: Causal mask + Padding mask (단방향)
#   - Cross-Attention: Padding mask
#
# GPT-1:
#   - Look-ahead (Causal) mask만 사용
#   - 모든 위치에서 미래 토큰 참조 방지
# ============================================================================

def generate_causal_mask(seq):
    """[GPT-1] Look-ahead (Causal) Mask 생성
    
    각 위치에서 이전 위치만 참조 가능 (미래 토큰 마스킹)
    GPT-1은 이 마스크만 사용하여 autoregressive 생성 수행
    
    Args:
        seq: (B, T) 입력 시퀀스
    Returns:
        mask: (B, 1, T, T) 어텐션 마스크
    """
    B, T = seq.shape
    # 패딩 마스크: PAD가 아닌 위치만 True
    pad_mask = (seq != PAD_ID).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, T)
    # [GPT-1] Look-ahead mask: 하삼각 행렬 (i번째 토큰은 j<=i만 참조)
    causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=seq.device))
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)
    # 패딩 AND 인과 마스크 결합
    return pad_mask & causal_mask

print('generate_causal_mask 함수 정의 완료')

### [변경사항] GPT-1 모델 전체 구조

**기존 Transformer**:
```
Encoder (소스) → Decoder (타겟) → 출력
```

**GPT-1**:
```
Token Embedding + Positional Embedding → GPT Blocks × N → 출력
```

In [ ]:
# ============================================================================
# [GPT-1 변경사항] 전체 모델 구조
# 
# 기존 Transformer:
#   class Transformer:
#       self.encoder = Encoder(...)  ← 제거됨
#       self.decoder = Decoder(...)
#       self.fc_out = nn.Linear(...)
#
# GPT-1:
#   class GPT1:
#       self.token_embed = nn.Embedding(...)
#       self.pos_embed = PositionalEmbedding(...)  # 학습 가능
#       self.blocks = nn.ModuleList([GPTBlock(...)])  # Decoder blocks
#       self.fc_out = nn.Linear(...)
# ============================================================================

class GPT1(nn.Module):
    """GPT-1 Model
    
    [Transformer 대비 주요 변경사항]
    1. 인코더 제거 → Decoder-only 구조
    2. Cross-Attention 제거 → Self-Attention만 사용
    3. Positional Encoding → Positional Embedding (학습 가능)
    4. ReLU → GELU 활성화 함수
    5. Look-ahead mask만 사용
    """
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout, vocab_size, max_len=512):
        super().__init__()
        
        # [GPT-1] Token Embedding
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        
        # [GPT-1 변경] Positional Embedding (기존: Positional Encoding)
        # 학습 가능한 위치 임베딩 사용
        self.pos_embed = PositionalEmbedding(max_len, d_model)
        
        # [GPT-1] GPT Blocks (기존 Decoder Layers와 유사하나 Cross-Attention 없음)
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        
        # 출력 레이어
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """[GPT-1] 단일 시퀀스 입력
        
        기존 Transformer: forward(src, tgt) - 소스와 타겟 분리
        GPT-1: forward(x) - 단일 시퀀스 ([BOS] + 소스 + [SEP] + 타겟 + [EOS])
        
        Args:
            x: (B, T) 입력 시퀀스
        Returns:
            logits: (B, T, vocab_size) 출력 로짓
            attns: 어텐션 가중치 리스트
        """
        # [GPT-1] Look-ahead mask 생성
        mask = generate_causal_mask(x).to(x.device)
        
        # Token Embedding + Positional Embedding
        tok_emb = self.token_embed(x)  # (B, T, d_model)
        pos_emb = self.pos_embed(tok_emb)  # (1, T, d_model)
        out = self.dropout(tok_emb + pos_emb)
        
        # GPT Blocks 통과
        attns = []
        for block in self.blocks:
            out, attn = block(out, mask)
            attns.append(attn)
        
        # 출력 로짓
        logits = self.fc_out(out)
        
        return logits, attns

print('GPT1 모델 클래스 정의 완료')

---
## Step 4. 훈련하기
### 1. 모델 선언

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('사용 디바이스:', device)

# [GPT-1] 모델 생성 (Encoder 없음, Decoder-only)
gpt_model = GPT1(
    n_layers   = 6,       # GPT-1 논문: 12 layers (여기서는 경량화)
    d_model    = 512,     # GPT-1 논문: 768
    n_heads    = 8,       # GPT-1 논문: 12
    d_ff       = 2048,    # GPT-1 논문: 3072
    dropout    = 0.1,
    vocab_size = tokenizer.GetPieceSize(),
    max_len    = MAX_LEN,
).to(device)

total_params = sum(p.numel() for p in gpt_model.parameters() if p.requires_grad)
print(f'총 파라미터 수: {total_params:,}')
print('\n=== GPT-1 모델 구조 ===')
print(gpt_model)

### 2. Learning Rate Scheduler & Adam Optimizer

In [ ]:
import torch.optim as optim

WARMUP_STEPS = 4000

class WarmupScheduler:
    def __init__(self, d_model, warmup_steps):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def get_lr(self):
        self.step_num += 1
        lr = (self.d_model ** -0.5) * min(
            self.step_num ** -0.5,
            self.step_num * (self.warmup_steps ** -1.5)
        )
        return lr

scheduler = WarmupScheduler(d_model=512, warmup_steps=WARMUP_STEPS)

optimizer = optim.Adam(
    gpt_model.parameters(),
    betas=(0.9, 0.98),
    eps=1e-9
)
print('Optimizer & Scheduler 준비 완료')

### 3. Loss 함수 정의

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, reduction='sum')

def compute_loss(pred, target):
    """
    [GPT-1] Language Model Loss 계산
    
    pred  : (B, seq_len, vocab_size)
    target: (B, seq_len)
    """
    pred = pred.reshape(-1, pred.size(-1))  # (B*seq_len, vocab_size)
    target = target.reshape(-1)              # (B*seq_len,)
    loss = criterion(pred, target)
    non_pad = target.ne(PAD_ID).sum().item()
    return loss / non_pad if non_pad > 0 else loss

### 4. train_step 함수 정의

In [ ]:
def train_step(seq, model, optimizer):
    """[GPT-1] 단일 시퀀스 학습
    
    기존 Transformer: train_step(src, tgt, model, optimizer)
    GPT-1: train_step(seq, model, optimizer) - 소스+타겟 결합된 단일 시퀀스
    """
    model.train()
    seq = seq.to(device)
    
    # [GPT-1] 입력: [:-1], 타겟: [1:] (다음 토큰 예측)
    seq_input = seq[:, :-1]   # 입력
    seq_target = seq[:, 1:]   # 정답 (한 칸 shift)
    
    # LR 업데이트
    lr = scheduler.get_lr()
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    
    optimizer.zero_grad()
    
    # [GPT-1] 단일 입력만 전달 (기존: model(src, tgt))
    pred, attns = model(seq_input)
    loss = compute_loss(pred, seq_target)
    loss.backward()
    optimizer.step()
    
    return loss, attns

### 5. 번역 함수

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

def translate(sentence, model, tok, max_len=50, plot_attention=False):
    """[GPT-1] 번역 함수
    
    입력 형식: [BOS] + 한국어 + [SEP] + (영어 생성)
    """
    model.eval()
    
    # [GPT-1] 입력 시퀀스 구성: [BOS] + 한국어 + [SEP]
    src_ids = tok.EncodeAsIds(clean_text(sentence))
    input_ids = [BOS_ID] + src_ids + [SEP_ID]
    
    # 영어 토큰 생성
    for _ in range(max_len):
        input_tensor = torch.tensor(input_ids).unsqueeze(0).to(device)
        with torch.no_grad():
            pred, attns = model(input_tensor)
        
        # 마지막 위치의 예측에서 다음 토큰 선택
        next_token = pred[0, -1].argmax().item()
        
        if next_token == EOS_ID:
            break
        input_ids.append(next_token)
    
    # SEP 이후의 토큰만 추출하여 디코딩
    sep_idx = len(src_ids) + 2  # BOS + 한국어 + SEP
    output_ids = input_ids[sep_idx:]
    translation = tok.DecodeIds(output_ids)
    
    print(f'  입력: {sentence}')
    print(f'  번역: {translation}\n')
    
    if plot_attention and attns:
        attn = attns[-1][0, 0].cpu().detach().numpy()
        plt.figure(figsize=(8, 6))
        plt.imshow(attn, cmap='viridis', aspect='auto')
        plt.colorbar()
        plt.title(f'GPT-1 Self-Attention — "{sentence}"')
        plt.xlabel('Key positions')
        plt.ylabel('Query positions')
        plt.tight_layout()
        plt.show()
    
    return translation

### 6. 전체 학습 루프

In [ ]:
import random
from tqdm import tqdm

BATCH_SIZE = 64
EPOCHS = 20

examples = [
    '오바마는 대통령이다.',
    '시민들은 도시 속에 산다.',
    '커피는 필요 없다.',
    '일곱 명의 사망자가 발생했다.',
]

print('=== GPT-1 모델 학습 시작 ===')
print(f'배치 크기: {BATCH_SIZE}, 에폭: {EPOCHS}')
print(f'데이터 수: {gpt_train.shape[0]:,}\n')

for epoch in range(EPOCHS):
    total_loss = 0
    idx_list = list(range(0, gpt_train.shape[0], BATCH_SIZE))
    random.shuffle(idx_list)
    t = tqdm(idx_list, desc=f'Epoch {epoch+1:2d}')
    
    for batch, idx in enumerate(t):
        # [GPT-1] 단일 시퀀스로 학습
        batch_loss, _ = train_step(
            gpt_train[idx:idx+BATCH_SIZE],
            gpt_model,
            optimizer
        )
        total_loss += batch_loss
        t.set_postfix({'Loss': f'{total_loss.item() / (batch+1):.4f}'})
    
    print(f'\n=== Epoch {epoch+1} 번역 결과 ===')
    for ex in examples:
        translate(ex, gpt_model, tokenizer)

---
## Attention Map 시각화

In [ ]:
translate('오바마는 대통령이다.', gpt_model, tokenizer, plot_attention=True)

---
## 최종 결과 요약

### GPT-1 vs Transformer 변경사항 정리

| 항목 | Transformer | GPT-1 |
|------|-------------|-------|
| 구조 | Encoder-Decoder | Decoder-only |
| Encoder | O | X (제거) |
| Cross-Attention | O | X (제거) |
| Self-Attention | Encoder/Decoder 각각 | 반복 사용 |
| Positional Info | Encoding (sin/cos) | Embedding (학습) |
| 활성화 함수 | ReLU | GELU |
| Masking | Encoder 양방향 + Decoder 단방향 | Look-ahead mask만 |
| 입력 형태 | src, tgt 분리 | 단일 시퀀스 |

### 하이퍼파라미터

| 파라미터 | 값 |
|----------|----|
| n_layers | 6 |
| d_model | 512 |
| n_heads | 8 |
| d_ff | 2048 |
| dropout | 0.1 |
| warmup_steps | 4000 |
| batch_size | 64 |
| epochs | 20 |